# Mortality Data Merger (2000–2016)
Merges individual yearly CSV files, cleans Crude Rate values, and computes per-county averages.

In [ ]:
import pandas as pd
from functools import reduce
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────
# Update DATA_DIR to the folder containing your CSV files
DATA_DIR = Path(".")   # e.g. Path("/home/jovyan/data")

# Files are named: Mortality_2000.csv, Mortality_2001.csv, ... Mortality_2016.csv
# Adjust the pattern below if your filenames differ
YEARS = range(2000, 2017)   # 2000 through 2016 inclusive
FILE_PATTERN = "Mortality_{year}.csv"   # change if needed

print(f"Looking for {len(list(YEARS))} files in: {DATA_DIR.resolve()}")

In [ ]:
# ── Load & clean each year ────────────────────────────────────────────────────
dfs = []
missing = []

for year in YEARS:
    filepath = DATA_DIR / FILE_PATTERN.format(year=year)

    if not filepath.exists():
        print(f"  WARNING: {filepath.name} not found — skipping")
        missing.append(year)
        continue

    temp = pd.read_csv(
        filepath,
        usecols=["County", "County Code", "Crude Rate"],
        dtype={"County": "str", "County Code": "str", "Crude Rate": "str"},
    )

    # Drop footer/notes rows (no County Code)
    temp = temp[temp["County Code"].notna() & (temp["County Code"].str.strip() != "")]

    # Clean Crude Rate: strip text like '(Unreliable)', convert to float
    temp["Crude Rate"] = (
        temp["Crude Rate"]
        .str.replace(r"[^\d.]", "", regex=True)   # keep digits and decimal point
        .replace("", float("nan"))                 # blank → NaN
        .astype(float)
    )

    temp = temp[["County", "County Code", "Crude Rate"]].rename(
        columns={"Crude Rate": f"crude_{year}"}
    )
    dfs.append(temp)
    print(f"  Loaded {year}: {len(temp):,} counties")

if missing:
    print(f"\nMissing years: {missing}")
print(f"\nLoaded {len(dfs)} year(s) successfully.")

In [ ]:
# ── Merge all years on County + County Code ───────────────────────────────────
final_df = reduce(
    lambda left, right: left.merge(right, on=["County", "County Code"], how="outer"),
    dfs,
)

print(f"Merged shape: {final_df.shape}")
final_df.head()

In [ ]:
# ── Compute average crude rate across all available years ─────────────────────
crude_cols = [c for c in final_df.columns if c.startswith("crude_")]

final_df["avg_crude_rate"] = final_df[crude_cols].mean(axis=1, skipna=True)

# Count how many years had valid (non-NaN) data per county
final_df["years_with_data"] = final_df[crude_cols].notna().sum(axis=1)

print("Average crude rate stats:")
print(final_df["avg_crude_rate"].describe())
final_df[["County", "County Code", "avg_crude_rate", "years_with_data"]].head(10)

In [ ]:
# ── Save output ───────────────────────────────────────────────────────────────
out_path = DATA_DIR / "mortality_merged_2000_2016.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()}")
print(f"Rows: {len(final_df):,} | Columns: {final_df.shape[1]}")